In [1]:
import pandas as pandas
import numpy as np 
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from Binning_Embedding_Bucketization import *
from Deep_Q_Learning import *
import os
from io import StringIO

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import joblib

X_train: torch.Size([70, 17, 17]), y_train: torch.Size([70, 17, 17])
X_val: torch.Size([15, 17, 17]), y_val: torch.Size([15, 17, 17])
X_test: torch.Size([15, 17, 17]), y_test: torch.Size([15, 17, 17])


In [2]:
in_path_t = "/media/arupreza/Assets/UIDS-II/Cognitive-Belief-Driven-Q-Learning-for-Vehicle-Model-Agnostic-Intrusion-Detection-in-V-Net/Input_data/Tesla/"

In [3]:
Tesla_Normal = pd.read_csv(in_path_t + "Tesla_Normal.csv")

Tesla_DoS_1 = pd.read_csv(in_path_t + "Tesla_DoS_1.csv")
Tesla_DoS_2 = pd.read_csv(in_path_t + "Tesla_DoS_2.csv")
Tesla_DoS_3 = pd.read_csv(in_path_t + "Tesla_DoS_3.csv")

Tesla_DoS = pd.concat([Tesla_DoS_1, Tesla_DoS_2, Tesla_DoS_3], axis=0)

Tesla_Fuzz_1 = pd.read_csv(in_path_t + "Tesla_Fuzz_1.csv")
Tesla_Fuzz_2 = pd.read_csv(in_path_t + "Tesla_Fuzz_2.csv")
Tesla_Fuzz_3 = pd.read_csv(in_path_t + "Tesla_Fuzz_3.csv")

Tesla_Fuzz = pd.concat([Tesla_Fuzz_1, Tesla_Fuzz_2, Tesla_Fuzz_3], axis=0)


Tesla_Rep_1 = pd.read_csv(in_path_t + "Tesla_Rep_1.csv")
Tesla_Rep_2 = pd.read_csv(in_path_t + "Tesla_Rep_2.csv")
Tesla_Rep_3 = pd.read_csv(in_path_t + "Tesla_Rep_3.csv")

Tesla_Rep = pd.concat([Tesla_Rep_1, Tesla_Rep_2, Tesla_Rep_3], axis=0)

In [4]:
def plot_distribution(df, column_name):
    """
    Plots the distribution of categories from the specified column in the dataframe.
    
    Parameters:
    df (pd.DataFrame): The DataFrame containing the data.
    column_name (str): The column name containing the categories to be plotted.
    """
    # Get the value counts and reset the index
    value_counts = df[column_name].value_counts().reset_index()

    # Rename the columns for clarity
    value_counts.columns = ['Category', 'Count']

    # Create a bar plot for the value counts of the specified column
    fig = px.bar(value_counts, 
                 x='Category', 
                 y='Count', 
                 labels={'Category': 'Category', 'Count': 'Count'},
                 title=f"Distribution of {column_name} Categories")

    # Show the plot
    fig.show()

In [5]:
Tesla_Normal = Tesla_Normal.fillna(0)
Tesla_Normal

,Time_Offset,ID,Data_Length,One,Two,Three,Four,Five,Six,Seven,Eight,Time_Gap
0,0.871,0370,8,12,01,08,1D,1F,BD,20,A7,0.000
1,1.167,0187,8,21,65,EA,00,60,EA,00,00,0.296
2,1.435,02D5,8,94,15,14,91,03,00,00,00,0.268
3,1.707,0186,8,8E,05,00,02,00,00,00,00,0.272
4,1.955,0385,8,70,6F,F6,FE,00,00,10,6B,0.248
...,...,...,...,...,...,...,...,...,...,...,...,...
599995,191614.045,02D5,8,C3,4F,14,86,03,00,00,00,0.248
599996,191614.281,0186,8,3D,0F,00,02,00,97,0E,00,0.236
599997,191614.515,0385,8,70,6F,F7,80,00,00,D0,AE,0.234
599998,191614.677,038B,4,61,4D,20,00,-1,-1,-1,-1,0.162


In [6]:
nan_count = Tesla_Normal.isna().sum().sum()  # Counts all NaN values in the DataFrame

print(f"Total number of NaN values in Tesla_Normal: {nan_count}")

Total number of NaN values in Tesla_Normal: 0


In [7]:
def cat_con(df):
    a = CAN_ID_Categorization(df["ID"])
    b = Time_Gap_Categorization(df["Time_Gap"])
    x = pd.concat([a, b], axis=1)
    return x

In [8]:
Tesla_Normal_Cat = cat_con(Tesla_Normal)
Tesla_Normal_Cat = Tesla_Normal_Cat[["Cat_CAN_ID", "Cat_Time_Gap"]]
Tesla_Normal_Cat

,Cat_CAN_ID,Cat_Time_Gap
0,CAN_46,TG_1
1,CAN_19,TG_65
2,CAN_38,TG_59
3,CAN_19,TG_59
4,CAN_48,TG_54
...,...,...
599995,CAN_38,TG_54
599996,CAN_19,TG_52
599997,CAN_48,TG_51
599998,CAN_48,TG_36


In [9]:
Tesla_DoS = Tesla_DoS.fillna(0)
Tesla_DoS_Cat = cat_con(Tesla_DoS)
Tesla_DoS_Cat = Tesla_DoS_Cat[["Cat_CAN_ID", "Cat_Time_Gap"]]
Tesla_DoS_Cat

,Cat_CAN_ID,Cat_Time_Gap
0,CAN_48,TG_1
1,CAN_21,TG_1
2,CAN_39,TG_1
3,CAN_21,TG_1
4,CAN_49,TG_1
...,...,...
1807047,CAN_49,TG_1
1807048,CAN_22,TG_1
1807049,CAN_1,TG_522
1807050,CAN_1,TG_924


In [10]:
Tesla_Fuzz = Tesla_Fuzz.fillna(0)
Tesla_Fuzz_Cat = cat_con(Tesla_Fuzz)
Tesla_Fuzz_Cat = Tesla_Fuzz_Cat[["Cat_CAN_ID", "Cat_Time_Gap"]]
Tesla_Fuzz_Cat

,Cat_CAN_ID,Cat_Time_Gap
0,CAN_48,TG_1
1,CAN_22,TG_1
2,CAN_39,TG_1
3,CAN_21,TG_1
4,CAN_49,TG_1
...,...,...
1807985,CAN_22,TG_1
1807986,CAN_46,TG_471
1807987,CAN_23,TG_849
1807988,CAN_31,TG_776


In [11]:
Tesla_Rep = Tesla_Rep.fillna(0)
Tesla_Rep_Cat = cat_con(Tesla_Rep)
Tesla_Rep_Cat = Tesla_Rep_Cat[["Cat_CAN_ID", "Cat_Time_Gap"]]
Tesla_Rep_Cat

,Cat_CAN_ID,Cat_Time_Gap
0,CAN_43,TG_1
1,CAN_20,TG_1
2,CAN_36,TG_1
3,CAN_20,TG_1
4,CAN_44,TG_1
...,...,...
1807109,CAN_50,TG_998
1807110,CAN_51,TG_412
1807111,CAN_14,TG_856
1807112,CAN_100,TG_578


In [12]:
# ----------------------------
# Build, Train (Optional), and Save
# ----------------------------
def build_and_save_embedding_model(df, model_path='can_embedding_model.pth', vocab_path='fixed_vocabs.pkl', train_epochs=3):
    # Skip categorization - you're already providing categorized values like 'CAN_46', 'TG_1'

    # Step 1: Fixed vocab mapping
    can_vocab = {f"CAN_{i+1}": i for i in range(100)}
    gap_vocab = {f"TG_{i+1}": i for i in range(1000)}

    df['CAN_ID_Encoded'] = df['Cat_CAN_ID'].map(can_vocab)
    df['Time_Gap_Encoded'] = df['Cat_Time_Gap'].map(gap_vocab)

    # Step 2: Initialize model
    model = CANEmbeddingModel(can_vocab_size=100, gap_vocab_size=1000)

    # Step 3: Optional training loop
    dataset = EmbeddingDataset(df)
    loader = DataLoader(dataset, batch_size=32, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    model.train()
    for epoch in range(train_epochs):
        for can_ids, gap_ids in loader:
            embeddings = model(can_ids, gap_ids)
            loss = embeddings.norm().mean() * 0  # dummy loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        print(f"✅ Epoch {epoch+1}/{train_epochs} completed")

    # Step 4: Save model and vocab
    torch.save(model.state_dict(), model_path)
    joblib.dump({'can_vocab': can_vocab, 'gap_vocab': gap_vocab}, vocab_path)

    print(f"✅ Embedding model saved to: {model_path}")
    print(f"✅ Fixed vocabularies saved to: {vocab_path}")
    return model

In [13]:
#model = build_and_save_embedding_model(Tesla_Normal_Cat)

In [14]:
def generate_embeddings_from_df(df, model_path='can_embedding_model.pth', vocab_path='Tesla_df_fixed.pkl'):
    # Load model
    model = CANEmbeddingModel()
    model.load_state_dict(torch.load(model_path))
    model.eval()

    # Load vocab
    vocabs = joblib.load(vocab_path)
    can_vocab = vocabs['can_vocab']
    gap_vocab = vocabs['gap_vocab']

    # Encode using vocab
    df['CAN_ID_Encoded'] = df['Cat_CAN_ID'].map(can_vocab)
    df['Time_Gap_Encoded'] = df['Cat_Time_Gap'].map(gap_vocab)

    # Convert to tensors
    can_ids = torch.tensor(df['CAN_ID_Encoded'].values, dtype=torch.long)
    gap_ids = torch.tensor(df['Time_Gap_Encoded'].values, dtype=torch.long)

    # Generate embeddings
    with torch.no_grad():
        embeddings = model(can_ids, gap_ids)

    # Optionally attach to DataFrame
    df['Embeddings'] = embeddings.tolist()

    # Print results
    print("🔍 Input DataFrame:")
    print(df[['Cat_CAN_ID', 'Cat_Time_Gap']])
    print("\n📦 Embeddings:")
    print(embeddings)

    return embeddings

In [15]:
embeddings = generate_embeddings_from_df(Tesla_Normal_Cat)

/tmp/ipykernel_15663/652111389.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path))


🔍 Input DataFrame:
       Cat_CAN_ID Cat_Time_Gap
0          CAN_46         TG_1
1          CAN_19        TG_65
2          CAN_38        TG_59
3          CAN_19        TG_59
4          CAN_48        TG_54
...           ...          ...
599995     CAN_38        TG_54
599996     CAN_19        TG_52
599997     CAN_48        TG_51
599998     CAN_48        TG_36
599999     CAN_20       TG_274

[600000 rows x 2 columns]

📦 Embeddings:
tensor([[ 0.8153,  0.7560, -0.1508,  ..., -0.0207, -0.2478,  0.0209],
        [-0.3260, -0.9409, -0.8805,  ..., -1.1633,  0.5448,  0.1928],
        [-0.7128,  1.3440, -1.5007,  ...,  1.8473, -0.4214,  1.9138],
        ...,
        [-0.4874, -0.3233, -0.7065,  ..., -0.4472,  0.9445,  1.3756],
        [-0.4874, -0.3233, -0.7065,  ..., -0.1318, -2.1114,  0.7585],
        [-0.1443,  0.2348, -1.1433,  ...,  0.0333, -0.4701,  0.2121]])


In [16]:
embeddings.shape

torch.Size([600000, 17])

In [17]:
pd.DataFrame(embeddings)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
0,0.815317,0.755972,-0.150839,-0.997748,-0.901599,0.749874,2.389574,-0.967674,0.636152,0.030871,0.349822,0.588009,1.741540,-1.435633,-0.020660,-0.247843,0.020950
1,-0.325961,-0.940922,-0.880549,-0.308889,-0.081851,-1.427086,-0.921703,-0.892342,-0.390798,-0.326480,-0.279746,-0.558007,2.360197,-2.166696,-1.163300,0.544805,0.192817
2,-0.712803,1.344026,-1.500708,0.943940,-0.682343,-0.778593,2.371649,0.129721,-0.656520,0.349087,-0.074838,1.010870,0.188213,-0.786195,1.847264,-0.421372,1.913787
3,-0.325961,-0.940922,-0.880549,-0.308889,-0.081851,-1.427086,2.371649,0.129721,-0.656520,0.349087,-0.074838,1.010870,0.188213,-0.786195,1.847264,-0.421372,1.913787
4,-0.487377,-0.323335,-0.706504,0.417193,1.138976,0.014613,-0.295372,1.391821,0.826395,-0.283504,0.442050,0.772759,0.393856,0.445890,-0.490602,-1.926068,-0.853287
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
599995,-0.712803,1.344026,-1.500708,0.943940,-0.682343,-0.778593,-0.295372,1.391821,0.826395,-0.283504,0.442050,0.772759,0.393856,0.445890,-0.490602,-1.926068,-0.853287
599996,-0.325961,-0.940922,-0.880549,-0.308889,-0.081851,-1.427086,0.364324,-1.269547,-0.135477,-0.061479,1.090131,0.313011,-0.682784,-0.323389,-0.592959,0.416132,0.622829
599997,-0.487377,-0.323335,-0.706504,0.417193,1.138976,0.014613,1.714351,-0.354135,-0.252150,-0.674452,0.019607,-0.181189,1.146424,-0.759419,-0.447195,0.944474,1.375642
599998,-0.487377,-0.323335,-0.706504,0.417193,1.138976,0.014613,1.530715,-0.894021,2.376017,-0.079577,0.202127,-0.519010,-0.899502,-0.112177,-0.131844,-2.111371,0.758544
